# 06.01 链表与连续数组的访存对比章节概述

本章节从链表、连续数组以及哈希冲突处理中的链地址法出发，对比“指针跳转式不规则访存”和“连续数组式规则访存”的实现差异，并以哈希查找中的桶化连续数组方案作为主要工程案例。

## 1. 本章前置要求

学习本章前，建议先具备以下基础：

1. 编程基础：能够阅读和运行 Python、C/C++ 代码，理解数组、循环、函数、结构体和文件读写。
2. 数据结构基础：理解数组、链表、哈希表、键值对、链地址法和开放寻址等基本概念。
3. 算法基础：了解查找问题的基本目标，知道冲突处理会影响平均访问次数和最坏访问路径。
4. 实验环境基础：能够使用 Jupyter Notebook（运行 Code Cell（代码单元格），并对 CANN（Compute Architecture for Neural Networks，异构计算架构）、Ascend C（昇腾 C 语言算子开发方式）和 NPU（Neural Processing Unit，神经网络处理器）算子开发流程有初步认识。

## 2. 本章主要内容

本章围绕“链表与连续数组的访存对比”展开。课程先从 CPU（Central Processing Unit，中央处理器）上常见的链表结构出发，说明链表插入灵活但节点地址不连续，遍历时需要根据 `next` 指针逐步跳转；随后对比连续数组的顺序布局、对齐搬运和分块访问，解释为什么它更适合 NPU 的 GM（Global Memory，全局内存）到 UB（Unified Buffer，统一缓冲区）搬运模型。并结合哈希冲突、链地址法、开放寻址与 Cuckoo Hash（布谷鸟哈希，一种为每个键设置少量候选位置的哈希表方法）的内容进行“链式冲突处理”与“桶化连续数组查找”的对比设计。最后通过可运行 Notebook 完成数据生成、连续桶化查询 Kernel（NPU 核函数）、C++ Host（主机侧程序）、构建运行、CPU Golden（CPU 参考结果）对比和综合练习。

### 2.1 连续数组与顺序访存

连续数组把同类型元素放在相邻地址上，例如 `values[i]` 与 `values[i+1]` 之间的地址差固定。对 NPU Kernel 来说，这种布局可以把一段数据按块从 GM 搬入 UB，再在 UB 中完成循环计算。数据访问越规则，越容易设计 `DataCopy`（Ascend C 数据搬运接口）、向量化计算、尾块处理和多核切分。

连续数组并不自动保证算法最快，但它让访存路径更稳定：访问范围、元素个数和字节数通常可以在 Host 侧提前推导。

### 2.2 链表与指针跳转

链表把每个节点分散存放，节点中保存 `next` 指针或下一个节点编号。它的优点是插入和删除灵活，不需要整体搬移元素；缺点是遍历时必须先读当前节点，才能知道下一步去哪里。这个“读出地址后再访问下一处”的过程会形成指针跳转，也叫 pointer chasing。

在 NPU 上，链表式访问通常难以一次性搬运连续大块数据：不同 query（查询请求）的链长不同、分支不同、下一节点地址也不同，容易造成控制流发散和 GM 随机小块访问。

In [ ]:
import numpy as np

# 连续数组：逻辑顺序和物理下标一致，适合分块搬运。
array_values = np.arange(8, dtype=np.int32) * 10
print("连续数组:", array_values.tolist())
print("连续读取 [2:6]:", array_values[2:6].tolist())

# 链表：next_idx 决定下一步访问位置，逻辑顺序不再等于物理下标顺序。
node_value = np.array([10, 40, 20, 70, 30], dtype=np.int32)
next_idx = np.array([2, -1, 4, 1, 3], dtype=np.int32)
order = []
cur = 0
while cur != -1:
    order.append(int(node_value[cur]))
    cur = int(next_idx[cur])
print("链表遍历顺序:", order)

### 2.3 哈希冲突处理中的链表与连续桶

哈希查找中的核心问题是哈希冲突：多个 key（键，查找时用于定位数据的标识）可能映射到同一位置。链地址法会在每个桶后挂一条链，查询时沿链逐个比较；开放寻址会在数组中继续探测；Cuckoo Hash（布谷鸟哈希）则让每个 key 拥有少量固定候选位置。

从访存角度看，链地址法体现了“链表式不规则访存”，桶化 Cuckoo Hash 体现了“连续数组式规则访存”。本实验采用每桶 4 个 `(key,value)` 的扁平布局：

```text
[key0,value0,key1,value1,key2,value2,key3,value3]
```

8 个 `int32` 正好是 32B。一次 query 固定读取两个候选桶，即 `2 × 32B = 64B`，再检查 8 个候选槽。

![连续桶化数组布局](images/cuckoo_bucket_layout.png)

### 2.4 从可变链长到固定桶访问

链表查询的访问次数由链长决定。若某个桶冲突严重，查询可能需要读很多个节点；不同 query 的循环次数也不同。连续桶化查询则把候选范围限制在固定数量的槽中，哪怕第一个桶已经命中，也可以继续检查第二个桶，使基础控制流程保持一致。

这种设计并不是说连续桶化在所有场景都绝对更快，而是更容易获得明确的访存上界、统一的分支路径和可预估的 UB 使用量。

In [ ]:
# 用访问次数对比链地址法和固定两桶访问。
chain_lengths = np.array([1, 3, 0, 6, 2], dtype=np.int32)
query_bucket = np.array([0, 1, 3, 4], dtype=np.int32)
linked_node_reads = int(chain_lengths[query_bucket].sum())

bucket_bytes = 32
fixed_bucket_reads = len(query_bucket) * 2
fixed_table_bytes = fixed_bucket_reads * bucket_bytes

print("链地址法节点读取次数:", linked_node_reads)
print("连续桶化读取桶次数:", fixed_bucket_reads)
print("连续桶化表数据字节数:", fixed_table_bytes)

### 2.5 NPU 映射与 UB 预算

Host 根据 query 数 Q（Query Count，查询数量）和可用 Vector Core（向量计算核心）数量确定实际启用的核心数，然后把连续 query 平均分配给各核心。每个核心按 `QUERY_TILE`（查询分块大小）分块搬入 query，再对每个 query 读取两个 32B 候选桶，在 UB 中比较 key 并连续写回 `values_out`（输出值数组）与 `found_out`（命中标记数组）。

链表方案如果直接搬到 NPU，会遇到三个问题：第一，下一节点地址依赖当前节点内容；第二，不同 query 的链长不同，核内循环次数不一致；第三，节点通常无法按 32B 或更大粒度连续搬运。连续桶化方案把主要随机访问压缩成固定两个桶，并让 query、输出数组保持连续分块。

![连续桶化查询的 Host 与 NPU 数据流](images/cuckoo_npu_dataflow.png)

### 2.6 复杂度与边界

从算法复杂度看，链地址法在装载均匀时平均查找可能接近 $O(1)$，但最坏情况会退化到一条长链；固定桶化查询把每个 query 的候选检查限制为 8 个槽，基础表访问量固定为 64B。工程实现时不能只看大 O 记号，还要关注地址是否连续、搬运是否对齐、分支是否稳定、尾块如何处理以及是否需要独立的 `found`（命中标记）标记。


### 2.7 课后练习

1. 为什么连续数组通常比链表更适合 NPU 的 GM 到 UB 分块搬运？
2. 链地址法处理哈希冲突时，为什么容易产生随机小块访存？
3. 本实验中的 4 槽桶化 Cuckoo Hash 体现了哪一类访存布局？
4. 为什么固定读取两个桶会增加少量冗余访问，却有利于并行实现？

运行下面的 Code Cell 可以查看参考答案。


In [ ]:
from pathlib import Path

p = Path("answer/06.01_chapter_intro/answers.md")
if not p.exists():
    p = Path("06_linked_list_vs_contiguous_array") / p
print(p.read_text(encoding="utf-8"))


## 3. 学习目标

完成本章后，学习者能够：

1. 说明链表和连续数组在访存路径、地址连续性和分块搬运上的差异。
2. 解释链地址法、开放寻址和桶化 Cuckoo Hash 在哈希冲突处理中的访存特征。
3. 推导固定两桶查询的候选槽数量、表访问字节数和多核切分参数。
4. 理解 query 连续分块、UB 缓存候选桶和输出连续写回在 NPU 算子中的作用。
5. 使用 CPU Golden 验证连续桶化查找结果，并分析链长、装载率、尾块和未命中标记等边界情况。

## 4. 小节简介与跳转链接

<table
  align="left"
  style="width: 80%;
         max-width: 1200px;
         margin: 0 auto 0 0 !important;
         text-align: left;">
  <thead>
    <tr>
      <th style="text-align: left;">小节</th>
      <th style="text-align: left;">简介</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td style="text-align: left;"><a href="./06.01_chapter_intro.ipynb" target="_self">06.01 链表与连续数组的访存对比章节概述</a></td>
      <td style="text-align: left;">介绍本章前置要求、主要内容，包括链表指针跳转、连续数组分块搬运、哈希冲突处理中的链地址法与桶化连续数组，以及学习目标和小节导航。</td>
    </tr>
    <tr>
      <td style="text-align: left;"><a href="./06.02_memory_access_compare.ipynb" target="_self">06.02 链表与连续数组访存对比实验</a></td>
      <td style="text-align: left;">通过可执行 Code Cell 完成连续桶化查找的数据生成、Tiling（数据切分）、Ascend C Kernel、C++ Host、CMake（跨平台构建工具）构建、NPU 运行和结果验证，并与链式访问模型对比。</td>
    </tr>
    <tr>
      <td style="text-align: left;"><a href="./06.03_chapter_practice.ipynb" target="_self">06.03 章节实践</a></td>
      <td style="text-align: left;">通过补全链表遍历、连续桶查询、多核 Tiling 和访存量估算，加深对数据布局与硬件映射的理解。</td>
    </tr>
  </tbody>
</table>
<div style="clear: both;"></div>